# Eukaryotic toehold switch — two-input AND

Manual test harness for `engine.gates.toehold.ToeholdAndGate`, built against a
**eukaryotic host** (`Host.HUMAN`/`Host.YEAST`, Kozak-cap-dependent-scanning
track — see `ToeholdGate`'s own docstring on why `host` is a constructor
parameter rather than a subclass). It inherits every method from `ToeholdGate`
except `generate_designs`, and sets `max_inputs = 2`. Scientific methods print
`pending Step 5` until the bodies land.

This notebook is `toehold_and.ipynb`'s sibling, not a replacement — that one
exercises the same stub against `Host.ECOLI` (steric RBS occlusion, serial
hairpins). Kept separate rather than parameterised over host in one notebook
because, once Step 5 lands, the two tracks are expected to diverge in real
construction (Kozak/scanning-blockage vs. RBS/steric-occlusion — see
`docs/modalities.md` and `ToeholdGate`'s own single-input eukaryotic work,
`toehold/eukaryotic-usage-example.ipynb`, for how much that single-input case
already had to diverge), not just in which fixed motif gets substituted in.

## The mechanism

An AND toehold opens only when **both** triggers are present. The
single-input eukaryotic case (`EukaryoticToeholdGate`, already built — see
`toehold/eukaryotic-usage-example.ipynb`) blocks a scanning 40S ribosome with a
closed hairpin sitting *upstream* of Kozak+AUG, not by sequestering Kozak
itself the way the prokaryotic RBS-in-loop layout sequesters the RBS. Extending
that to two inputs raises questions the prokaryotic serial-stem shape doesn't
have to answer, none of them resolved here — this notebook watches for them
once Step 5 lands, it doesn't decide them:

* **Where do two hairpins sit relative to one Kozak?** The prokaryotic AND
  puts the start codon inside the *inner* of two nested hairpins, with an RBS
  free in each hairpin's own loop. A `"trailing"`-style eukaryotic construct has
  no RBS-in-loop to nest a second one inside — the open question is whether the
  outer hairpin blocks scanning *before* it ever reaches an inner hairpin+Kozak,
  or whether Kozak itself gets sequestered by the inner stem the way the
  prokaryotic AUG does today.
* **Order still matters** — A-outer/B-inner is a different construct from the
  reverse, same as the prokaryotic case.
* **The intermediate (one-trigger) state is real** — if a scanning ribosome can
  already reach Kozak+AUG with only the first trigger present, the gate is an OR
  wearing an AND's shape. Evaluate the single-trigger states explicitly once
  `evaluate_design` exists for this class.
* **Kozak self-binding risk, doubled.** The single-input case already tracks
  whether a trigger-derived toehold happens to carry Kozak's reverse complement
  (`_kozak_rc_in_toehold`, and its worst-case `predicted_leakage` penalty in
  `evaluate_design` — eukaryotic-only, gated on `self.host.track`). An AND gate
  has two trigger-derived toeholds and one Kozak copy; whichever construction
  Step 5 picks needs the same check run against *both*, not just the one
  nearest Kozak in sequence.

## Setup

In [ ]:
# Put the shared _fixtures.py on the path. It lives in the notebooks/ root, one
# level up from this gate's folder; search upward so the notebook works wherever
# Jupyter is launched. fx.bootstrap() then adds <repo>/src so `import engine...`
# resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / '_fixtures.py').exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()

## Build the gate

In [ ]:
host = fx.Host.HUMAN  # HUMAN | YEAST both take the eukaryotic (Kozak) track
gate = fx.toehold_and(host=host)
fx.describe_gate(gate)

## Inputs

A `TriggerSet` (the circuit's inputs) and a `Constraints` (the researcher's
limits). Both are made up here — override any field via keyword.

In [ ]:
triggers = fx.sample_trigger_set(n_activators=2)   # two activators; may share a gene
constraints = fx.sample_constraints(max_switch_length=240)

for t in triggers.activators:
    print(f'{t.trigger_id}  {t.symbol:6}  {t.sequence}  {t.length} nt')
print('arity:', triggers.arity, '| logic:', triggers.logic_type)

## `required_tools()` — implemented

In [ ]:
fx.attempt('required_tools', gate.required_tools)

## `is_compatible()` *(Step 5)*

Try it with a one-activator set too — a two-input family should reject arity 1
with a message written for the researcher.

In [ ]:
fx.attempt('is_compatible (2 activators)', lambda: gate.is_compatible(triggers, constraints))
one = fx.sample_trigger_set(n_activators=1)
fx.attempt('is_compatible (1 activator)', lambda: gate.is_compatible(one, constraints))

## `generate_designs()` *(Step 5)*

Expect both trigger orders to be generated, and the single-trigger states
evaluated explicitly. For this host, also expect whichever layout(s) Step 5
settles on for a eukaryotic AND to be swept the way the single-input gate
sweeps `kozak_layouts` — not silently fixed to one shape.

In [ ]:
designs = fx.attempt(
    'generate_designs',
    lambda: list(gate.generate_designs(triggers, constraints)),
)

## `evaluate_design()` *(Step 5)*

In [ ]:
target = designs[0] if designs else fx.sample_design(gate, triggers)
fx.attempt('evaluate_design', lambda: gate.evaluate_design(target))

## `emit_sequence()` and `describe()` — output helpers

These two are implemented today. `generate_designs()` is not, so
`fx.sample_design(...)` hands us a plausible `GateDesign` to call them on.

In [ ]:
design = fx.sample_design(gate, triggers)
print('design_id  ', design.design_id)
print('gate_kind  ', design.gate_kind)
print('length     ', design.length, 'nt')
print('emit_sequence:', gate.emit_sequence(design))
print('describe     :', gate.describe(design))

## Switching in real folding

Everything above runs against `fx.StubFoldEngine` — deterministic, fake, no
ViennaRNA. For genuine structure predictions, pass `real_fold=True` when you build
the gate (needs `import RNA` to work in this environment):

```python
gate = fx.toehold_and(host=host, real_fold=True)
folder = fx.fold_engine(real=True)
folder.mfe('GGGAAACCCUUUGGGAAACCC')   # -> FoldResult(structure, energy)
```